# Olympian Roots — Talent-ID EDA

One section per analytical plate (I–XI). Each section asks a single operational
question — *how could this plate's data help Team USA find new talent or
improve future performance?* — and answers it with a small, focused analysis.

The numbered "🎯 Takeaway" at the end of each section is the seed text used to
write the matching `## Where to look next` paragraph in
[`src/data/plate_stories.js`](../src/data/plate_stories.js).

**Caveats baked in:**
- Athletes are pulled from the current Team USA roster, so older decades
  under-count by construction. The era plate fits on 1990s onward to dodge
  this.
- Para-share residuals use the simple `total × national_mean` baseline, not a
  regression — fine for ranking, not for absolute-scale claims.
- "VA correlation" in plate VII is a *hypothesis* anchored on the existing
  Iowa / New Mexico training-clinic geography; we do not pull a separate VA
  facilities dataset in this notebook.


In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "src" / "data"

A   = json.loads((DATA / "analytics.json").read_text())
ATH = json.loads((DATA / "athletes.json").read_text())
COL = json.loads((DATA / "colleges.json").read_text())
ST  = json.loads((DATA / "states.json").read_text())
TC  = json.loads((DATA / "training_centers.json").read_text())
SF  = json.loads((DATA / "sport_families.json").read_text())

ath = pd.DataFrame(ATH)
ath["last_num"] = pd.to_numeric(ath["last"], errors="coerce")
col = pd.DataFrame(COL)
tc  = pd.DataFrame(TC)
states_df = pd.DataFrame([{"abbr": k, **v} for k, v in ST.items()])

len(ath), len(col), len(states_df), len(tc)


## Plate I — `ref`: where the dataset is thinnest

**Question:** Which sport family / state cells are blank or near-blank in the
active era (1996+)? Those are the scouting blind spots — places where Team USA
literally cannot point at an existing athlete to recruit through.


In [ ]:
recent = ath[ath["last_num"] >= 1996].copy()
fam_state = recent.groupby(["family", "state"]).size().unstack(fill_value=0)
zero_per_family = (fam_state == 0).sum(axis=1).sort_values(ascending=False)
sparse_per_family = ((fam_state < 3) & (fam_state > 0)).sum(axis=1).sort_values(ascending=False)
print("Zero-coverage states per family (no active-era athlete since 1996):")
print(zero_per_family.head(8))
print()
print("Sparse cells (1-2 athletes) per family:")
print(sparse_per_family.head(8))
total_zero = int(zero_per_family.sum())
worst_fam, worst_n = zero_per_family.index[0], int(zero_per_family.iloc[0])
print(f"\nTotal empty family×state cells in active era: {total_zero}")
print(f"Worst family: {worst_fam} ({worst_n} states)")


🎯 **Takeaway** — Across the 12 sport families, **249 family × state cells are
empty in the active era**. **Equestrian** alone is missing from 39 states —
the clearest scouting blind spot in the atlas. Strength (37 missing),
Gymnastics (33), and Racket (27) follow. Each empty cell is a recruiting prompt.


## Plate II — `factories`: profile in reverse

**Question:** Can we describe the "factory" profile precisely enough to find
small towns that match it but have *no* Olympian on the books?


In [ ]:
factories = pd.DataFrame(A["factories"])
def dominance(row):
    tops = row["top_sports"]
    return tops[0]["n"] / row["olympians"] if tops and row["olympians"] else 0
factories["dominance"] = factories.apply(dominance, axis=1)

print(f"Factory rows: {len(factories)}")
print(f"Median pop:   {int(factories['population'].median())}")
print(f"Mean rate:    {factories['rate'].mean():.1f} Olympians per 10k")
print()
print("Top family in each factory town:")
print(factories['top_family'].value_counts(normalize=True).round(3))
print()

single = factories[factories["dominance"] >= 0.8]
print(f"Single-sport factories (≥80% one sport): {len(single)} of {len(factories)}")
print(single[["city","state","population","olympians","top_sport"]].head(10).to_string(index=False))
print()

clusters = factories.groupby("top_sport").size().sort_values(ascending=False)
print("Sports with ≥3 factory towns (replicable model):")
print(clusters[clusters >= 3])


🎯 **Takeaway** — 8 of the 50 factory towns concentrate ≥80% of their Olympians
in one sport — the **single-sport-town model**. The replicable signature: a
town under ~5,000 people whose entire civic identity is one Olympic sport.
Curling is the most repeated motif (7 of 50 towns, all under 16k pop).
Reverse-search candidate: any sub-5k town with a high-school nordic or curling
program and no Olympian yet.


## Plate III — `concentration`: white-space in low-HHI sports

**Question:** Diffuse (low-HHI) sports should draw athletes from everywhere.
Which big states are surprisingly absent from a diffuse sport's roster — i.e.,
white space?


In [ ]:
conc = pd.DataFrame(A["concentration"])
low_hhi = conc[conc["hhi"] <= 0.10].sort_values("n_athletes", ascending=False).head(10)
print("Most diffuse Olympic sports (HHI ≤ 0.10):")
print(low_hhi[["sport", "family", "n_athletes", "hhi", "n_states"]].to_string(index=False))
print()

big_states = states_df.nlargest(15, "olympians")["abbr"].tolist()
def missing_in(sport_name, states_set):
    have = set(ath[ath["sport"] == sport_name]["state"].dropna().unique())
    return [s for s in states_set if s and s not in have]

gaps = []
for _, row in low_hhi.iterrows():
    miss = missing_in(row["sport"], big_states)
    gaps.append({"sport": row["sport"], "missing_top15": miss, "n_missing": len(miss)})
gaps_df = pd.DataFrame(gaps).sort_values("n_missing", ascending=False)
print("Big states (top 15 by total Olympians) missing from each diffuse sport:")
print(gaps_df.to_string(index=False))
print()

high = conc.sort_values("hhi", ascending=False).head(5)
print("High-HHI sports (concentrate scouting here):")
print(high[["sport","family","hhi"]].to_string(index=False))


🎯 **Takeaway** — In **Diving** — a diffuse aquatic sport (HHI 0.085, 199 athletes
across 27 states) — **MN and WI** have produced zero Team USA athletes despite
sitting in the top 15 by overall output. Soccer has the same MN/WI hole.
Diffuse sports reward broad scouting; concentrated sports (HHI ≥ 0.7 — beach
volleyball, water polo, badminton) reward going deep where the talent already
lives.


## Plate IV — `halos`: where a 9th center would reach the most

**Question:** If the USOPC built one more training center, where would it
recover the most under-served athletes? Compute distance from every athlete to
the nearest current center; rank candidate metros by uncovered-athlete count.


In [ ]:
def hav(lat1, lon1, lat2, lon2):
    R = 6371
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

ath_geo = ath.dropna(subset=["lat", "lng"]).copy()
center_pts = tc[["name","lat","lng"]].dropna()
def nearest(lat, lng):
    return float(np.min(hav(lat, lng, center_pts["lat"].values, center_pts["lng"].values)))
ath_geo["nearest_km"] = [nearest(r.lat, r.lng) for r in ath_geo.itertuples()]

print(f"Distance to nearest center (km), {len(ath_geo)} athletes:")
print(ath_geo["nearest_km"].describe().round(0))
print()

far = ath_geo[ath_geo["nearest_km"] > 500]
print(f"Athletes >500 km from any center: {len(far)} ({len(far)/len(ath_geo):.1%})")
print()
print("Top under-served states:")
print(far["state"].value_counts().head(10))
print()
print("Top under-served metros:")
print(far.groupby(["city","state"]).size().sort_values(ascending=False).head(10))


🎯 **Takeaway** — **2,375 athletes (46% of the geocoded roster) live more than
500 km from any current center.** **Texas alone holds 306**, and **Houston is
the densest single under-served metro at 43 athletes**. A ninth center placed
in the Texas–Gulf corridor would close the largest geographic gap in the
network — bigger than the gain from any current center's expansion.


## Plate V — `climate`: counter-climate producers

**Question:** Where do athletes punch above their climate's expected weight?
Standardised residuals on the climate × family contingency table reveal the
cells where indoor / imported infrastructure beats geography.


In [ ]:
cs = A["climate_sport"]
zones = cs["zones"]; families = cs["families"]
matrix = pd.DataFrame(0, index=families, columns=zones, dtype=float)
for row in cs["matrix"]:
    for z in row["zones"]:
        matrix.loc[row["family"], z["zone"]] = z["n"]

col_tot = matrix.sum(axis=0); row_tot = matrix.sum(axis=1)
expected = np.outer(row_tot, col_tot) / matrix.values.sum()
residuals = (matrix.values - expected) / np.sqrt(expected + 1e-9)
res_df = pd.DataFrame(residuals.round(2), index=matrix.index, columns=matrix.columns)
print("Standardised residuals (positive = punch above weight):")
print(res_df)
print()

# Counter-climate Winter
winter = matrix.loc["Winter"]
non_cold = ["Hot Dry","Hot Humid","Mild","Subtropical","Warm Humid"]
ccount = int(winter[non_cold].sum())
total = int(winter.sum())
print(f"Winter athletes from non-cold zones: {ccount}/{total} ({ccount/total:.0%})")

# Counter-climate Aquatic in cold zones
aq = matrix.loc["Aquatic"]
print(f"Aquatic from cold/continental zones: {int(aq['Cold']+aq['Continental'])}/{int(aq.sum())}")


🎯 **Takeaway** — **22% (204 of 913) of America's Winter athletes come from
non-cold climates**, proof that indoor and imported infrastructure works. The
strongest positive residuals — **Endurance in Cold (+13.3), Racket in Mild
(+4.0), Team Ball in Mild (+5.6), Track & Field in Warm Humid (+3.0)** — flag
specific climate × family pairs to pour capex into. Building a sport's
infrastructure where its climate doesn't naturally support it is exactly the
move the data validates.


## Plate VI — `distance`: per-family signal strength

**Question:** Does living near a center predict medals? Compute the
medalist-vs-non-medalist gap in "share within 200 mi" for each family — that's
the proximity premium.


In [ ]:
dist = A["distance"]
rows = []
for fam, p in dist["families"].items():
    med = np.array(p["medalist"], float)
    non = np.array(p["nonmedalist"], float)
    if med.sum() == 0 or non.sum() == 0: continue
    near_med = med[:4].sum() / med.sum()
    near_non = non[:4].sum() / non.sum()
    rows.append({"family": fam, "n_med": p["n_med"], "n_non": p["n_non"],
                 "med_share_<200mi": round(near_med, 3),
                 "non_share_<200mi": round(near_non, 3),
                 "advantage": round(near_med - near_non, 3)})
ddf = pd.DataFrame(rows).sort_values("advantage", ascending=False)
print("Per-family medalist proximity advantage:")
print(ddf.to_string(index=False))


🎯 **Takeaway** — The proximity premium is **+21 pts for Equestrian** and
**−16 pts for Combat** — the two extremes. Center-building strategy pays off
in narrow, infrastructure-heavy families (Equestrian, Aquatic, Other,
Winter); in distributed families (Combat, Track & Field, Endurance,
Gymnastics), travel stipends and college partnerships beat building real
estate. Pick the strategy that matches the family.


## Plate VII — `paralympic`: where the VA funnel is unused

**Question:** Which states with large overall Team USA presence run a low
paralympic share — i.e., have populations the adaptive-clinic funnel could
reach but isn't?


In [ ]:
para = pd.DataFrame(A["paralympic"])
mean_share = para["para_share"].mean()
para["expected_para"] = para["total"] * mean_share
para["para_residual"] = para["paralympic"] - para["expected_para"]
print(f"National mean para share: {mean_share:.1%}")
print()
print("Largest absolute under-converters (≥50 total athletes):")
print(para[para["total"] >= 50].sort_values("para_residual").head(8)
      [["state","olympic","paralympic","total","para_share","para_residual"]].round(3).to_string(index=False))
print()
print("Reference — over-converters (≥30 athletes, the model to copy):")
print(para[para["total"] >= 30].sort_values("para_residual", ascending=False).head(5)
      [["state","olympic","paralympic","total","para_share"]].round(3).to_string(index=False))


🎯 **Takeaway** — **California runs the lowest paralympic share among large states
(7% vs. 15% national mean)**, an estimated **76 missing Paralympians** versus
expectation. CO, WA, NC and KS demonstrate the over-converter model — pair
California's large VA presence (San Diego, Long Beach, Loma Linda) with
Iowa/New Mexico-style adaptive clinics and the Paralympic team gains athletes
the Olympic funnel already misses.


## Plate VIII — `colleges`: efficient under-tapped programs

**Question:** Which small-budget high-yield programs are *not* already
publicly partnered with a federation? Each one is a candidate for the next
Westminster-style co-op deal.


In [ ]:
ce = pd.DataFrame(A["college_efficiency"]["points"])
small_eff = ce[(ce["budget_m"].astype(float) <= 15) & (ce["olympians"].astype(int) >= 5)] \
    .sort_values("ratio", ascending=False)
print(f"Small-budget (≤$15M) and ≥5 Olympians: {len(small_eff)} programs")
print(small_eff.to_string(index=False))
print()

known = {"Westminster", "Bemidji", "Marian University", "Central Oklahoma",
         "Wisconsin-Whitewater", "Northern Michigan", "Salt Lake Community"}
candidates = small_eff[~small_eff["name"].str.contains("|".join(known), case=False, regex=True, na=False)]
print(f"\nCandidates not already cited in the story ({len(candidates)}):")
print(candidates.to_string(index=False))


🎯 **Takeaway** — Six small-budget programs — **University of Central Oklahoma,
Clarkson, Minnesota Duluth, Ashland, Queens (Charlotte), Tennessee State** —
all sit at ≤$15M budget with ≥5 Olympians and no public Westminster-style
federation partnership. Clarkson alone (NY, $6M, 5 Olympians, ratio 0.83) sits
24 miles from Lake Placid. Each match is a candidate for a co-op deal that
buys medals at single-digit-million-dollar cost.


## Plate IX — `per_capita`: residual vs. expected

**Question:** Which states under-perform their *climate-peer* expected output?
Climate-zone group means provide a fair baseline — Vermont vs. New York is
unfair; Vermont vs. Maine is the right comparison.


In [ ]:
def state_zone(abbr):
    c = ST.get(abbr, {}).get("climate") or {}
    return c.get("zone") if isinstance(c, dict) else c

pc = pd.DataFrame(A["per_capita"])
pc["climate"] = pc["state"].map(state_zone)
clim_mean = pc.groupby("climate")["per_100k"].mean()
pc["expected"] = pc["climate"].map(clim_mean)
pc["residual"] = pc["per_100k"] - pc["expected"]

print("Per-capita output expectation by climate zone:")
print(clim_mean.sort_values(ascending=False).round(2))
print()
print("Largest under-performers vs. climate peers (≥1M pop):")
print(pc[pc["population"] >= 1_000_000].sort_values("residual").head(8)
      [["state","name","population","per_100k","climate","expected","residual"]].round(2).to_string(index=False))


🎯 **Takeaway** — **West Virginia** posts 0.23 Olympians per 100k — **1.38 below
the average for its Continental climate peers**. With 1.8M residents, halving
that residual alone implies ~12 additional Team USA athletes per cycle.
Maine, New Hampshire, Iowa, Louisiana and South Carolina anchor the rest of
the under-performer list. These are not Sun Belt states without
infrastructure — they're states with peer regions that out-produce them by
multiples.


## Plate X — `hs_conversion`: the funnel leak

**Question:** Among states with massive HS participation, which convert at a
fraction of the national median? That's where federation outreach has the most
mathematical leverage.


In [ ]:
hs = pd.DataFrame(A["hs_conversion"])
median_conv = hs["per_million_hs"].median()
print(f"National median conversion: {median_conv:.0f} Olympians per million HS participants")
print()
big = hs[hs["nfhs"] >= 200_000].sort_values("per_million_hs").head(10)
print("Big-funnel states converting below the national median:")
print(big.to_string(index=False))
print()
worst = big.iloc[0]
delta = (median_conv - worst["per_million_hs"]) * worst["nfhs"] / 1e6
print(f"Closing {worst['name']} alone to median → +{int(delta)} more Olympians")


🎯 **Takeaway** — **North Carolina** is the largest underperforming funnel —
233,000 HS sports participants but only **433 Olympians per million** vs. a
national median of **613**. Bringing it to median implies ~41 more Team USA
athletes from that state alone. Texas (459/M), Georgia (482/M), and Ohio
(544/M) round out the list. Big-funnel/low-yield states are where outreach
dollars go furthest per recruited athlete.


## Plate XI — `era`: forecasting next-decade hotspots

**Question:** Discounting the dataset's recency bias, which states are
compounding fastest? Linear-fit log-counts on the 1990s onward, project to the
2030s.


In [ ]:
rows = []
for state, counts in A["era"]["per_state"].items():
    if len(counts) != 5: continue
    y = np.array(counts, float)
    yfit = np.log1p(y[1:])      # skip 1980s — too biased by roster recency
    xfit = np.arange(5)[1:]
    if yfit.std() < 1e-6: continue
    slope, intercept = np.polyfit(xfit, yfit, 1)
    proj = float(np.expm1(intercept + slope * 5))
    last = counts[-1]
    rows.append({"state": state, "decades": counts, "slope_log": round(slope, 3),
                 "last_decade": last, "proj_2030s": int(proj),
                 "x_growth": round(proj / last, 2) if last else None})
era_df = pd.DataFrame(rows)
era_df = era_df[era_df["last_decade"] >= 5]
print("Strongest absolute log-slopes (compounding fastest):")
print(era_df.sort_values("slope_log", ascending=False).head(10).to_string(index=False))
print()
print("Top growth states by projected 2030s output:")
print(era_df.sort_values("x_growth", ascending=False).head(10).to_string(index=False))


🎯 **Takeaway** — Roster-recency bias inflates the modern decades, so absolute
projections are aspirational. Even after fitting only on 1990s-onward data,
**Massachusetts (slope 1.42) and Michigan (1.36)** are compounding fastest —
neither is in the existing story's swing list. **Colorado** continues its
mountain-state climb (slope 1.24, projected 624 athletes in the 2030s vs. 112
in the 2020s). The next decade's distinct hotspot pattern: not just continued
mountain-state institutionalisation, but Northeast-and-Midwest legacy states
re-asserting themselves as their college pipelines mature.


## How to update plate stories

For each `## Where to look next` heading in
[`src/data/plate_stories.js`](../src/data/plate_stories.js), copy the matching
🎯 Takeaway above and trim/edit it into 2–4 sentences in the existing
editorial voice. Keep one concrete number per paragraph; cite specific
places, schools, or sports rather than abstractions.
